# 1767. Find the Subtasks That Did Not Execute
**Level:** Hard

## Question
We need to report the IDs of the **missing subtasks** for each `task_id`.  

- Each task is divided into subtasks labeled from `1` to `subtasks_count`.  
- The `Executed` table shows which subtasks were successfully executed.  
- If a subtask does not appear in the `Executed` table, it is considered **missing**.  
- Return all missing subtasks for each task.  
- The result can be in any order.

---

## Schema

### Table: Tasks
| Column Name    | Type | Description                                  |
|----------------|------|----------------------------------------------|
| task_id        | INT  | Primary key, ID of the task                  |
| subtasks_count | INT  | Number of subtasks for the task (2–20 range) |

---

### Table: Executed
| Column Name | Type | Description                                  |
|-------------|------|----------------------------------------------|
| task_id     | INT  | Foreign key referencing Tasks                |
| subtask_id  | INT  | ID of the executed subtask                   |

**Primary Key:** (task_id, subtask_id)

---

## Sample Data

### Tasks
| task_id | subtasks_count |
|---------|----------------|
| 1       | 3              |
| 2       | 2              |
| 3       | 4              |

### Executed
| task_id | subtask_id |
|---------|------------|
| 1       | 2          |
| 3       | 1          |
| 3       | 2          |
| 3       | 3          |
| 3       | 4          |

### Expected Result
| task_id | subtask_id |
|---------|------------|
| 1       | 1          |
| 1       | 3          |
| 2       | 1          |
| 2       | 2          |

---



In [0]:
## PySpark Code to Create Schema, Data, and Temp Views

from pyspark.sql.types import StructType, StructField, IntegerType

# Schema for Tasks
tasks_schema = StructType([
    StructField("task_id", IntegerType(), False),
    StructField("subtasks_count", IntegerType(), False)
])

# Schema for Executed
executed_schema = StructType([
    StructField("task_id", IntegerType(), False),
    StructField("subtask_id", IntegerType(), False)
])

# Data for Tasks
tasks_data = [
    (1, 3),
    (2, 2),
    (3, 4)
]

# Data for Executed
executed_data = [
    (1, 2),
    (3, 1),
    (3, 2),
    (3, 3),
    (3, 4)
]

# Create DataFrames
tasks_df = spark.createDataFrame(tasks_data, tasks_schema)
executed_df = spark.createDataFrame(executed_data, executed_schema)

# Register Temp Views
tasks_df.createOrReplaceTempView("Tasks")
executed_df.createOrReplaceTempView("Executed")

# Quick check
tasks_df.show()
executed_df.show()


In [0]:
%sql
WITH all_subtasks AS (
  SELECT
    t.task_id,
    subtask_id
  FROM Tasks t
  LATERAL VIEW explode(sequence(1, t.subtasks_count)) AS subtask_id
)
SELECT a.task_id, a.subtask_id
FROM all_subtasks a
LEFT JOIN Executed e
  ON a.task_id = e.task_id
 AND a.subtask_id = e.subtask_id
WHERE e.task_id IS NULL
ORDER BY a.task_id, a.subtask_id;




## Setup in SQL server

```sql
-- Drop tables if they already exist
IF OBJECT_ID('Executed', 'U') IS NOT NULL DROP TABLE Executed;
IF OBJECT_ID('Tasks', 'U') IS NOT NULL DROP TABLE Tasks;

-- Create Tasks table
CREATE TABLE Tasks (
    task_id INT PRIMARY KEY,
    subtasks_count INT NOT NULL CHECK (subtasks_count BETWEEN 2 AND 20)
);

-- Create Executed table
CREATE TABLE Executed (
    task_id INT NOT NULL,
    subtask_id INT NOT NULL,
    PRIMARY KEY (task_id, subtask_id),
    FOREIGN KEY (task_id) REFERENCES Tasks(task_id)
);

-- Insert sample data into Tasks
INSERT INTO Tasks (task_id, subtasks_count) VALUES
(1, 3),
(2, 2),
(3, 4);

-- Insert sample data into Executed
INSERT INTO Executed (task_id, subtask_id) VALUES
(1, 2),
(3, 1),
(3, 2),
(3, 3),
(3, 4);



--code

	with cte as (
	select task_id , subtasks_count from tasks
	union all
	select task_id , subtasks_count -1 from cte where subtasks_count >1

	)
	select c.task_id as task_id , c.subtasks_count as subtask_id from cte c left join 
	Executed e on 
	e.task_id = c.task_id
	and 
	e.subtask_id = c.subtasks_count
	where e.task_id is NULL  
	order by c.task_id , c.subtasks_count 

# Databricks SQL: Recursive CTEs, LATERAL VIEW, and Array Functions

## Recursive CTEs
- **SQL Server / PostgreSQL**: Support recursive CTEs (`WITH RECURSIVE` or `WITH ...` in SQL Server).
- **Databricks SQL (Spark SQL)**: Does **not** support recursive CTEs.
  - You cannot reference a CTE inside itself.
  - Instead, use **array functions** (`sequence`, `explode`, `posexplode`, `inline`) to generate rows.

---

## LATERAL VIEW
- `LATERAL VIEW` is used in Spark SQL to **flatten arrays or maps** into rows.
- It comes **after the FROM clause** because it acts like a table generator:
  - Takes each row from the base table.
  - Applies a generator function (`explode`, `posexplode`, `inline`).
  - Produces additional rows that are “lateral” (side by side) with the original row.

Example:
```sql
SELECT t.task_id, subtask_id
FROM Tasks t
LATERAL VIEW explode(sequence(1, t.subtasks_count)) AS subtask_id;




## explode vs posexplode vs inline

### 1. `explode()`
- **Purpose**: Flattens an array into multiple rows.
- **Output**: Each element of the array becomes a row.
- **Use case**: When you only need the values.
- Example:
  ```sql
  SELECT explode(array(10,20,30)) AS val;
  -- Output: 10, 20, 30
  ```

### 2. `posexplode()`
- **Purpose**: Flattens an array into multiple rows **and includes the position index**.
- **Output**: Two columns → position + value.
- **Use case**: When you need both the element and its index (e.g., subtask IDs starting from 1).
- Example:
  ```sql
  SELECT posexplode(array('a','b','c')) AS (pos, val);
  -- Output: (0,'a'), (1,'b'), (2,'c')
  ```

### 3. `inline()`
- **Purpose**: Flattens an array of structs into multiple rows.
- **Output**: Each struct’s fields become columns.
- **Use case**: When you have arrays of complex objects (structs) and want to expand them into rows.
- Example:
  ```sql
  SELECT inline(array(struct(1,'x'), struct(2,'y')));
  -- Output: (1,'x'), (2,'y')
  ```

---

## When to Use
- **Use `explode`**: When you only care about the values in an array.
- **Use `posexplode`**: When you need both the values and their positions (indexes).
- **Use `inline`**: When dealing with arrays of structs and you want to flatten them into rows with multiple columns.

---

## Key Takeaway
- Databricks SQL does not support recursive CTEs.
- Use `sequence` + `explode`/`posexplode`/`inline` with `LATERAL VIEW` to generate rows.
- Choose the right function based on whether you need:
  - Just values (`explode`),
  - Values + positions (`posexplode`),
  - Expanded structs (`inline`).
```

---

This summary gives you a **ready-to-use documentation block**. Would you like me to also add a **visual diagram (in Markdown arrows)** showing the flow: `Tasks → sequence() → explode/posexplode → missing subtasks`? That would make it even easier for readers to grasp.